# 25 · Parallel tool calls
Level L2 · Part 3 · core · ~35 min · builds on 22

## Goal
When the model asks for several tools in one turn, run them at the same time with `asyncio`, send every
result back in one message with its own id, and know when to make the model ask for one tool at a time.

## The idea
In lesson 22 the model asked for one tool. Ask Scout to "compare the notes on Mars and Venus" and a
model can ask for **two** searches in the same reply: these are **parallel tool calls**. They do not
depend on each other, so your code can run them at the same time instead of one after the other. Then
all the results go back together, in **one** message, each tagged with the tool call id it answers.
Think of a waiter who takes a table's whole order at once: the kitchen cooks the dishes side by side,
and they all arrive together, each on the right plate. Sometimes you do want one dish at a time (the
second depends on the first), and then you turn parallel calls off.

## Picture
```mermaid
sequenceDiagram
    participant S as Scout (your code)
    participant M as Model
    S->>M: "Compare Mars and Venus" + tool schema
    M-->>S: call_1 search("Mars"), call_2 search("Venus")
    par at the same time
        Note over S: run search("Mars")
    and
        Note over S: run search("Venus")
    end
    S->>M: ONE message: result call_1 + result call_2
    M-->>S: final answer (text)
```

## Step by step
### 1. Start the recorder and set up
The recorder first, as in lesson 22. Without a key, the Gemini calls replay from
`../data/llm_cassettes/25_parallel-tool-calls.yaml`.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("25_parallel-tool-calls")

LLM mode: replay (25_parallel-tool-calls.yaml)


Setup from lessons 01, 07 and 22: the notes, the client, the pinned model, `search_notes`, its schema
and the Gemini tool config, unchanged.

In [2]:
import asyncio, json, os, time
from dotenv import load_dotenv
from google import genai
from google.genai import types

NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)

def search_notes(query: str) -> list[dict]:
    """Return the notes whose text contains `query` (ignoring case)."""
    q = query.lower()
    return [note for note in NOTES if q in note["text"].lower()]

TOOLS = {"search_notes": search_notes}
print("model:", MODEL, "| tools:", list(TOOLS))

model: gemini-3.5-flash-lite | tools: ['search_notes']


The schema and Gemini config from lesson 22:

In [3]:
SEARCH_NOTES_SCHEMA = {
    "name": "search_notes",
    "description": "Search the user's notes. Returns every note whose text contains the query word "
                   "(case-insensitive), as a list of {id, text}. Use it before answering any question "
                   "about what the notes say.",
    "parameters": {
        "type": "object",
        "properties": {"query": {"type": "string", "description": "one word to look for, e.g. 'Mars'"}},
        "required": ["query"],
    },
}
search_decl = types.FunctionDeclaration(name=SEARCH_NOTES_SCHEMA["name"],
    description=SEARCH_NOTES_SCHEMA["description"], parameters_json_schema=SEARCH_NOTES_SCHEMA["parameters"])
TOOL_CFG = types.GenerateContentConfig(tools=[types.Tool(function_declarations=[search_decl])],
                                       automatic_function_calling=NO_AFC, max_output_tokens=300)
print("config ready:", [d.name for d in TOOL_CFG.tools[0].function_declarations])

config ready: ['search_notes']


### 2. A fake model that asks for several tools at once
Lesson 22's fake asked for one search. This one asks for one search per planet it finds in the
question. Results now go back in **one** `tool` message holding a `results` list (lesson 22's single
result is the one-item case). The fake enforces the rule real APIs enforce: every call id from its last
turn answered, in one message. With `parallel=False` it asks for one tool per turn instead.

In [4]:
PLANETS = ["Mars", "Venus", "Jupiter", "Saturn"]

def check_tool_answers(messages: list[dict]) -> None:
    """If the model's last turn asked for tools, require ONE tool message answering every call id."""
    last = max(i for i, m in enumerate(messages) if m["role"] != "tool")
    if messages[last]["role"] != "assistant":
        return
    wanted = sorted(c["id"] for c in messages[last]["tool_calls"])
    after = messages[last + 1:]
    if len(after) != 1:
        raise ValueError(f"expected ONE tool message answering {wanted}, got {len(after)}")
    got = sorted(r["call_id"] for r in after[0]["results"])
    if got != wanted:
        raise ValueError(f"tool results {got} do not answer calls {wanted}")

The model itself: check the answers, ask for every planet not searched yet (or just the first one when
`parallel=False`), and answer once nothing is left.

In [5]:
class FakeParallelModel:
    def __init__(self, parallel: bool = True):
        self.parallel, self.calls_made, self.turns = parallel, 0, 0

    def __call__(self, messages: list[dict], tools: list[dict]) -> dict:
        self.turns += 1
        check_tool_answers(messages)
        question = next(m["text"] for m in messages if m["role"] == "user")
        asked = {c["args"]["query"] for m in messages if m["role"] == "assistant" for c in m.get("tool_calls", [])}
        todo = [p for p in PLANETS if p in question and p not in asked]
        if todo:
            calls = []
            for word in (todo if self.parallel else todo[:1]):
                self.calls_made += 1
                calls.append({"id": f"call_{self.calls_made}", "name": "search_notes", "args": {"query": word}})
            return {"role": "assistant", "tool_calls": calls}
        found = {r["name"] + ":" + r["query"]: len(json.loads(r["content"]))
                 for m in messages if m["role"] == "tool" for r in m["results"]}
        return {"role": "assistant", "text": f"Compared: {found}"}

### 3. Two tool calls in one turn
Ask the comparison question. The reply carries a **list** of tool calls, each with its own id.

In [6]:
fake = FakeParallelModel()
QUESTION = "Compare the notes on Mars and Venus."
messages = [{"role": "user", "text": QUESTION}]
reply = fake(messages, tools=[SEARCH_NOTES_SCHEMA])
for call in reply["tool_calls"]:
    print(call)

{'id': 'call_1', 'name': 'search_notes', 'args': {'query': 'Mars'}}
{'id': 'call_2', 'name': 'search_notes', 'args': {'query': 'Venus'}}


Running them in order works, and for a search over six notes it is instant. But real tools are slow:
a web search or a database query takes hundreds of milliseconds. Let's pretend `search_notes` takes
half a second and time the plain loop.

In [7]:
def slow_search_notes(query: str) -> list[dict]:
    time.sleep(0.5)                                  # pretend this is a network call
    return search_notes(query)

start = time.perf_counter()
results = [slow_search_notes(**c["args"]) for c in reply["tool_calls"]]
print(f"one after the other: {time.perf_counter() - start:.1f} s for {len(results)} calls")

one after the other: 1.0 s for 2 calls


Two calls, two waits added up: 1.0 s. With five calls it would be 2.5 s, while the CPU sits idle the
whole time, just waiting. That idle waiting is what `asyncio` lets us overlap.

### 4. An asyncio primer in four cells
A function defined with `async def` is a **coroutine function**. Calling it does not run it; it returns
a coroutine object, a paused piece of work.

In [8]:
async def slow_search_async(query: str) -> list[dict]:
    await asyncio.sleep(0.5)                         # wait WITHOUT blocking: other work may run now
    return search_notes(query)

coro = slow_search_async("Mars")
print(type(coro).__name__)
coro.close()                                         # we never ran it; close it to avoid a warning

coroutine


`await` runs a coroutine and waits for its result. Something has to drive coroutines: the **event
loop**. Jupyter already runs one, so in a notebook you can `await` at the top level of a cell. (In a
script you would write `asyncio.run(main())`.)

In [9]:
hits = await slow_search_async("Mars")
print("Mars ->", [n["id"] for n in hits])

Mars -> [1, 3, 5]


`await` one after another is still sequential. To overlap the waits, hand several coroutines to
`asyncio.gather`: the event loop starts them all, and while one is waiting in `asyncio.sleep` (or on the
network) the others make progress. `gather` returns the results **in the order you passed them in**, not
the order they finished.

In [10]:
start = time.perf_counter()
mars, venus = await asyncio.gather(slow_search_async("Mars"), slow_search_async("Venus"))
print(f"gather: {time.perf_counter() - start:.1f} s | Mars {[n['id'] for n in mars]} | Venus {[n['id'] for n in venus]}")

gather: 0.5 s | Mars [1, 3, 5] | Venus [2]


Half the time. Most tools you already have are plain functions (`def`, not `async def`), like our
`slow_search_notes` that calls `time.sleep`. `asyncio.to_thread` runs a plain function in a worker
thread and gives you something you can `await`, so plain tools join `gather` too.

In [11]:
start = time.perf_counter()
both = await asyncio.gather(asyncio.to_thread(slow_search_notes, "Mars"),
                            asyncio.to_thread(slow_search_notes, "Venus"))
print(f"to_thread + gather: {time.perf_counter() - start:.1f} s | {[len(r) for r in both]} notes")

to_thread + gather: 0.5 s | [3, 1] notes


### 5. Run all of a turn's tool calls at once
Now the helper Scout will use. It takes the model's tool calls, runs them concurrently and returns one
result per call, carrying that call's id. Because `gather` keeps input order, `zip` pairs every result
with the right call.

In [12]:
SLOW_TOOLS = {"search_notes": slow_search_notes}

async def run_tool_calls(tool_calls: list[dict], tools: dict) -> list[dict]:
    outputs = await asyncio.gather(*(asyncio.to_thread(tools[c["name"]], **c["args"]) for c in tool_calls))
    return [{"call_id": c["id"], "name": c["name"], "query": c["args"].get("query"),
             "content": json.dumps(out)} for c, out in zip(tool_calls, outputs)]

start = time.perf_counter()
tool_results = await run_tool_calls(reply["tool_calls"], SLOW_TOOLS)
print(f"{time.perf_counter() - start:.1f} s")
for r in tool_results:
    print(r["call_id"], r["query"], "->", [n["id"] for n in json.loads(r["content"])])

0.5 s
call_1 Mars -> [1, 3, 5]
call_2 Venus -> [2]


### 6. All results back in one message
Append the model's turn, then **one** tool message with every result. The fake checks that each call id
is answered, then writes its answer.

In [13]:
messages.append(reply)
messages.append({"role": "tool", "results": tool_results})
final = fake(messages, tools=[SEARCH_NOTES_SCHEMA])
print(final["text"])
print("history roles:", [m["role"] for m in messages], "| model turns:", fake.turns)

Compared: {'search_notes:Mars': 3, 'search_notes:Venus': 1}
history roles: ['user', 'assistant', 'tool'] | model turns: 2


Two model turns for two searches, same as lesson 22's single search.

### 7. The same thing with Gemini
Ask Gemini the comparison question with the lesson 22 config.

In [14]:
contents = [types.Content(role="user", parts=[types.Part(text=QUESTION)])]
resp1 = client.models.generate_content(model=MODEL, contents=contents, config=TOOL_CFG)
for fc in resp1.function_calls:
    print("tool call:", fc.id, fc.name, dict(fc.args))

tool call: call_2227560 search_notes {'query': 'Mars'}
tool call: call_2227561 search_notes {'query': 'Venus'}


Two function calls in one reply, each with its own id. Run them concurrently with the same pattern:
turn Gemini's calls into our dicts and reuse `run_tool_calls`.

In [15]:
calls = [{"id": fc.id, "name": fc.name, "args": dict(fc.args)} for fc in resp1.function_calls]
start = time.perf_counter()
g_results = await run_tool_calls(calls, SLOW_TOOLS)
print(f"ran {len(g_results)} tools in {time.perf_counter() - start:.1f} s")

ran 2 tools in 0.5 s


Gemini's version of "one message": a single `user` turn with one `function_response` part per call,
each with the call's id and name. The model's own turn goes in first, unchanged (lesson 22).

In [16]:
contents.append(resp1.candidates[0].content)
contents.append(types.Content(role="user", parts=[
    types.Part(function_response=types.FunctionResponse(
        id=r["call_id"], name=r["name"], response={"result": json.loads(r["content"])}))
    for r in g_results]))
resp2 = client.models.generate_content(model=MODEL, contents=contents, config=TOOL_CFG)
print("tool calls:", resp2.function_calls)
print(resp2.text)

tool calls: None
Based on the notes, here is a comparison between Mars and Venus:

* **Moons:** Mars has two small moons (Phobos and Deimos), whereas no moons are mentioned for Venus.
* **Rotation / Day Length:** A day on Mars lasts about 24 hours and 39 minutes, while Venus is unique for spinning backwards compared to most planets.
* **Space Exploration:** The rover Perseverance landed on Mars in February 2021, whereas no exploration missions are mentioned for Venus.


### 8. Turning parallel calls off
Parallel calls are wrong when one call needs another's result ("find the newest note, then summarise
it"), when order matters for side effects (write a file, then read it), or when a tool cannot take two
requests at once. Then you want one tool per turn. Here is the cost, with the fake:

In [17]:
for parallel in (True, False):
    m = FakeParallelModel(parallel=parallel)
    msgs = [{"role": "user", "text": QUESTION}]
    while "tool_calls" in (r := m(msgs, tools=[SEARCH_NOTES_SCHEMA])):
        msgs += [r, {"role": "tool", "results": [{"call_id": c["id"], "name": c["name"], "query": c["args"]["query"],
                     "content": json.dumps(search_notes(**c["args"]))} for c in r["tool_calls"]]}]
    print(f"parallel={parallel!s:5} -> model turns: {m.turns}, history length: {len(msgs)}")

parallel=True  -> model turns: 2, history length: 3
parallel=False -> model turns: 3, history length: 5


One extra model turn per extra tool: more latency and more input tokens, since every turn resends the
history (lesson 08). So turn parallel calls off only when you need to.

How to turn them off depends on the vendor. Anthropic and OpenAI have a switch (request payloads shown
by hand, from RESEARCH.md; lesson 23 covers the wire formats):

In [18]:
anthropic_request = {"model": "claude-sonnet-5-5", "max_tokens": 300, "tools": ["..."],
                     "tool_choice": {"type": "auto", "disable_parallel_tool_use": True}}
openai_request = {"model": "gpt-6-luna", "input": "...", "tools": ["..."], "parallel_tool_calls": False}
print(json.dumps(anthropic_request["tool_choice"]), "|", "parallel_tool_calls:", openai_request["parallel_tool_calls"])

{"type": "auto", "disable_parallel_tool_use": true} | parallel_tool_calls: False


Gemini's `generate_content` has no such switch: `ToolConfig` and `FunctionCallingConfig` offer only
`mode` and `allowed_function_names`. You can check the installed SDK yourself:

In [19]:
print("FunctionCallingConfig fields:", list(types.FunctionCallingConfig.model_fields))
print("any 'parallel' field:", any("parallel" in f for c in (types.GenerateContentConfig, types.ToolConfig,
      types.FunctionCallingConfig) for f in c.model_fields))

FunctionCallingConfig fields: ['allowed_function_names', 'mode', 'stream_function_call_arguments']
any 'parallel' field: False


So on Gemini you ask in the system instruction, and see whether it listens.

In [20]:
ONE_AT_A_TIME = TOOL_CFG.model_copy(update={"system_instruction":
    "Call at most one tool per turn. Wait for its result before calling the next tool."})
resp_one = client.models.generate_content(model=MODEL, contents=contents[:1], config=ONE_AT_A_TIME)
print("tool calls in this turn:", [(fc.name, dict(fc.args)) for fc in resp_one.function_calls])

tool calls in this turn: [('search_notes', {'query': 'Mars'})]


This time Gemini listened and asked for Mars only. But an instruction is a request, not a guarantee.
When order truly matters, also enforce it in your own
code: run the calls one after another, in the order given, instead of with `gather`. Every call still
gets its result, so the id rule holds.

In [21]:
async def run_tool_calls_in_order(tool_calls: list[dict], tools: dict) -> list[dict]:
    out = []
    for c in tool_calls:                             # await each one before starting the next
        res = await asyncio.to_thread(tools[c["name"]], **c["args"])
        out.append({"call_id": c["id"], "name": c["name"], "query": c["args"].get("query"), "content": json.dumps(res)})
    return out

start = time.perf_counter()
ordered = await run_tool_calls_in_order(reply["tool_calls"], SLOW_TOOLS)
print(f"in order: {time.perf_counter() - start:.1f} s, ids {[r['call_id'] for r in ordered]}")

in order: 1.0 s, ids ['call_1', 'call_2']


## At work
- **Parallel calls cut round trips, not tokens.** Vendors enable them by default; tool time becomes the
  slowest call instead of the sum. → lesson 26 (the agent loop), lesson 38 (many conversations at once)
- **One failing tool must not sink the others.** Plain `gather` raises the first exception and loses the
  other results; production loops return each error as that call's result. → lesson 28
- **Side-effecting tools run one at a time.** Teams set `disable_parallel_tool_use` /
  `parallel_tool_calls=false`, or serialise in code, for writes, payments and anything order-dependent.
  → lesson 52 (human approval for side-effecting tools)
- **"All results in one message" is a tested rule** (Anthropic: every `tool_result` block in one user
  message, before any text), covered by fake-model unit tests like ours. → lesson 46

## What just happened
- The model asked for two tools in one turn: **parallel tool calls**, each with its own tool call id.
- `async def` makes a coroutine; `await` runs one; the **event loop** drives them; `asyncio.gather`
  overlaps their waits and returns results in input order; `asyncio.to_thread` lets plain functions join.
- `run_tool_calls` ran a turn's calls concurrently: 0.5 s instead of 1.0 s.
- All results went back in **one** message (Gemini: one `user` turn, one `function_response` part per id).
- Turning parallel off costs an extra model turn per tool. Anthropic and OpenAI have a switch; Gemini
  needs an instruction, and order-sensitive code should also run calls in order itself.

## Common mistakes
**1. One message per result.** It feels natural to answer each call as it finishes, with its own
message. The APIs expect one message holding them all (Anthropic's docs: every `tool_result` block in a
single user message). Our fake enforces that rule, so the bug shows up at once.

In [22]:
m = FakeParallelModel()
msgs = [{"role": "user", "text": QUESTION}]
msgs.append(m(msgs, tools=[SEARCH_NOTES_SCHEMA]))
for r in tool_results:
    msgs.append({"role": "tool", "results": [r]})    # one message per result
m(msgs, tools=[SEARCH_NOTES_SCHEMA])

ValueError: expected ONE tool message answering ['call_1', 'call_2'], got 2

Fix: collect every result first, then send a single message.

In [23]:
msgs = msgs[:2] + [{"role": "tool", "results": tool_results}]
print(m(msgs, tools=[SEARCH_NOTES_SCHEMA])["text"])

Compared: {'search_notes:Mars': 3, 'search_notes:Venus': 1}


**2. `asyncio.run` inside Jupyter.** Code copied from a script calls `asyncio.run(...)`. Jupyter already
runs an event loop, and `asyncio.run` refuses to start a second one.

In [24]:
asyncio.run(run_tool_calls(reply["tool_calls"], SLOW_TOOLS))

RuntimeError: asyncio.run() cannot be called from a running event loop

Fix: in a notebook, `await` directly. In a `.py` script, where no loop is running yet,
`asyncio.run(main())` is right.

In [25]:
fixed = await run_tool_calls(reply["tool_calls"], SLOW_TOOLS)
print([r["call_id"] for r in fixed])

['call_1', 'call_2']


## Try it
Make `run_tool_calls` survive a failing tool. Add a tool `flaky_search(query)` that raises
`RuntimeError("search backend down")` when `query == "Venus"` and otherwise behaves like
`slow_search_notes`. Using `asyncio.gather(..., return_exceptions=True)`, return a result for every
call, where a failed call's `content` is `json.dumps({"error": str(exc)})`. Run it on the fake model's
two calls and print each `call_id` with its content.

In [26]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [27]:
llm_replay.stop()

## Key terms
- **parallel tool calls** — several tool calls in one model reply, each with its own tool call id; all their results go back in one message.
- **coroutine** — what calling an `async def` function returns: paused work that runs only when awaited or scheduled on the event loop.
- **event loop** — the asyncio scheduler that runs coroutines and switches between them whenever one is waiting.
- **`asyncio.gather`** — runs several awaitables concurrently and returns their results in the order they were passed in.
- **`asyncio.to_thread`** — runs a plain (blocking) function in a worker thread so it can be awaited alongside coroutines.
- **disable parallel tool use** — asking the model for at most one tool call per turn: Anthropic `disable_parallel_tool_use`, OpenAI `parallel_tool_calls=false`, Gemini only by instruction.